Reviewing Complexity of Movie Reviews

**Research question:** Is a more complex movie review (longer, harder to read) associated with a more polarized review score?

## How to run this notebook

**Input:** `rotten_tomatoes_data_with_vars.csv.zip`, the zipped cleaned dataset (120,622 Rotten Tomatoes critic reviews from 2021 onward).
It is in the GitHub repo.

**Steps:** In Colab, choose *Runtime → Run all*. When Section 1 asks for a file, upload the zip. Do not unzip it first.

**Outputs:**

| Output | Where it appears | What it is |
|---|---|---|
| Summary statistics | Section 3, printed + `OUTPUT/summary_statistics.csv` | Mean, spread and range of score, polarization, word count and grade level |
| `eda_distributions.png` | Section 3 | Histograms of word count and polarization |
| `eda_polarization_by_length.png` | Section 3 | Average polarization at each review length |
| `eda_grade_level.png` | Section 3 | Histogram of Flesch-Kincaid grade level |
| `rotten_tomatoes_data_with_vars.csv` | Section 3, downloaded | Cleaned data plus the new feature columns |
| Coefficient table | Section 4, printed + `OUTPUT/coefficient_table.csv` | Regression coefficients, robust SEs, p-values, 95% CIs, standardized betas |
| Performance table | Section 4, printed + `OUTPUT/performance_table.csv` | Held-out MAE, RMSE and R² vs. the baseline |
| VIF + diagnostics tables | Section 4, printed + `OUTPUT/vif_table.csv`, `OUTPUT/diagnostic_table.csv` | Multicollinearity and assumption checks |
| `model_diagnostics.png` | Section 4 | Residual, Q-Q, observed-vs-predicted and coefficient plots |
| Model testing tables | Section 5, printed + `OUTPUT/performance.csv`, `generalization.csv`, `criteria.csv`, `margin_table.csv` | Training vs. held-out results and the plan's success criteria |
| `model_testing.png` | Section 5 | Held-out residuals, predictions, errors and R² vs. goal |
| `OUTPUT.zip` | Section 6, downloaded | Every table (CSV) and graph (PNG) above in one file |

---
## Section 1: Load the cleaned data

Upload the zipped cleaned dataset and read it into a pandas DataFrame called `df`.

**Input:** `rotten_tomatoes_data_with_vars.csv.zip`
**Output:** `df` with one row per review; its size and first five rows are printed.

In [ ]:
# Open a file picker in Colab. Choose rotten_tomatoes_data_with_vars.csv.zip.
from google.colab import files

cleaned_data = files.upload()   # dict of {file name: file contents in bytes}

In [ ]:
import io
import pandas as pd

# Take the uploaded file's name and its raw bytes.
movie_data = next(iter(cleaned_data))
file_bytes = io.BytesIO(cleaned_data[movie_data])

# pandas can read a zipped CSV directly; the "compression" argument tells it
# the file is zipped (needed because we read from memory, not from a file path).
# A plain .csv still works too, in case someone uploads the unzipped file.
if movie_data.lower().endswith(".zip"):
    df = pd.read_csv(file_bytes, compression="zip")
else:
    df = pd.read_csv(file_bytes)

# OUTPUT: number of reviews and columns, then a preview of the data.
print(f"Loaded {movie_data}: {df.shape[0]:,} reviews x {df.shape[1]} columns")
df.head()

---
## Section 2: Create the model variables

Build the response variable (polarization) and the two complexity predictors (word count and Flesch-Kincaid grade level).

- **normalized_score**: `standardizedScore` (0-100) rescaled to 0-1.
- **polarization**: distance of the normalized score from the midpoint 0.5. It ranges from 0 (a middle score) to 0.5 (a perfect or zero score).
- **word_count**: number of words in the review.
- **grade_level**: Flesch-Kincaid grade level, i.e. the school grade needed to read the review.

**Output:** four new columns added to `df`; a preview of each is printed.

In [ ]:
# Rescale the 0-100 standardized score to 0-1.
df["normalized_score"] = df["standardizedScore"] / 100

# Polarization = how far the score is from the middle of the scale (0.5),
# so very high and very low scores are both "polarized".
df["polarization"] = (df["normalized_score"] - 0.5).abs()

# OUTPUT: preview of the score and the two new variables.
df[["standardizedScore", "normalized_score", "polarization"]].head()

In [ ]:
# Word count: split each review on spaces and count the pieces.
df["word_count"] = df["reviewText"].str.split().str.len()

# OUTPUT: preview of review text with its word count.
df[["reviewText", "word_count"]].head()

In [ ]:
# Install textstat, the package that calculates the Flesch-Kincaid grade level.
%pip install textstat

In [ ]:
import textstat

# Flesch-Kincaid grade level for every review (higher = harder to read).
# This takes a minute or two for ~120,000 reviews.
df["grade_level"] = df["reviewText"].apply(textstat.flesch_kincaid_grade)

# OUTPUT: preview of review text with its grade level.
df[["reviewText", "grade_level"]].head()

---
## Section 3: Exploratory data analysis

Check the distributions of the new variables for unusual values, and look at the raw relationship between review length and polarization before modeling.

**Outputs:** summary statistics table; `eda_distributions.png`, `eda_polarization_by_length.png`, `eda_grade_level.png`; and `rotten_tomatoes_data_with_vars.csv` (data with the new columns), which is downloaded.

In [ ]:
# OUTPUT: count, mean, standard deviation, min, quartiles and max of each variable.
# Use this to spot implausible values (e.g. negative or very high grade levels).
print("SUMMARY STATISTICS")
print(df[["normalized_score", "polarization", "word_count", "grade_level"]].describe())

In [ ]:
import matplotlib.pyplot as plt

# OUTPUT: histograms of review length and polarization (saved as eda_distributions.png).
df[["word_count", "polarization"]].hist(bins=25, figsize=(10, 4))
plt.suptitle("Distributions of review length and polarization")
plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=150)
plt.show()

# EDA: Movie reviews show a median length of 24 words with polarization values
# concentrated just under 0.1 and above 0.3.

In [ ]:
# Average polarization for each review length, to see if longer reviews are
# more or less polarized.
# OUTPUT: line plot (saved as eda_polarization_by_length.png).
df.groupby("word_count")["polarization"].mean().plot(
    xlabel="word count",
    ylabel="average polarization",
    title="Average polarization by length of movie review",
    figsize=(10, 4),
)
plt.tight_layout()
plt.savefig("eda_polarization_by_length.png", dpi=150)
plt.show()

# Average polarization looks to be around 0.2 across movie review lengths.
# Appears to have a weak relationship.

In [ ]:
# Distribution of grade level. The x-axis is limited to -10..30 so a few extreme
# values don't squash the plot (those extremes are handled in Section 4).
# OUTPUT: histogram (saved as eda_grade_level.png).
df["grade_level"].hist(bins=30, range=(-10, 30))
plt.title("Flesch-Kincaid grade level of reviews")
plt.xlabel("grade level")
plt.ylabel("number of reviews")
plt.tight_layout()
plt.savefig("eda_grade_level.png", dpi=150)
plt.show()

In [ ]:
# Save the data with the new feature columns. It is reloaded in Section 5 and
# downloaded so the team has a copy.
# OUTPUT: rotten_tomatoes_data_with_vars.csv (downloaded to your computer).
df.to_csv("rotten_tomatoes_data_with_vars.csv", index=False)
print(f"Saved rotten_tomatoes_data_with_vars.csv ({len(df):,} rows)")

from google.colab import files
files.download("rotten_tomatoes_data_with_vars.csv")

---
## Section 4: Model development
Fit a multiple linear regression predicting polarization from review length, readability and top-critic status, then check the model's assumptions.

**Model:** `polarization = b0 + b1*log(1 + word_count) + b2*grade_level + b3*is_top_critic`

**Outputs:** coefficient table, held-out performance table, VIF table, diagnostics table, and `model_diagnostics.png`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson, jarque_bera


# --------------------------------------------------------------------------
# 4.1 Build the modeling dataset
# --------------------------------------------------------------------------
# Remove duplicate review IDs and rows missing a required modeling variable.
model_data = df.drop_duplicates(subset=["reviewId"]).copy()

# Turn the True/False top-critic column into 1/0 so it can go in the regression.
model_data["is_top_critic"] = (
    model_data["isTopCritic"].astype(str).str.strip().str.lower().eq("true").astype(int)
)

# Winsorize continuous predictors at the 1st and 99th percentiles to reduce
# the influence of implausible/extreme values while retaining observations.
# (Values below the 1st percentile are raised to it; values above the 99th are lowered to it.)
for source, target in [("word_count", "word_count_model"),
                       ("grade_level", "grade_level_model")]:
    lower, upper = model_data[source].quantile([0.01, 0.99])
    model_data[target] = model_data[source].clip(lower, upper)

# Log-transform word count because its effect is likely to level off for long reviews.
model_data["log_word_count"] = np.log1p(model_data["word_count_model"])

# Keep only the response and the three predictors, and drop any rows with missing values.
features = ["log_word_count", "grade_level_model", "is_top_critic"]
model_data = (
    model_data[["polarization", *features]]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

In [ ]:
# --------------------------------------------------------------------------
# 4.2 Reproducible 80/20 train-test split
# --------------------------------------------------------------------------
# 80% of reviews are used to fit the model; 20% are held out to test it on
# data it has never seen. random_state=42 makes the split the same every run.
train_df, test_df = train_test_split(
    model_data,
    test_size=0.20,
    random_state=42,
)

# X = predictors (plus a constant for the intercept), y = polarization.
X_train = sm.add_constant(train_df[features], has_constant="add")
X_test = sm.add_constant(test_df[features], has_constant="add")
y_train = train_df["polarization"]
y_test = test_df["polarization"]

# OUTPUT: how many reviews are in each part of the split.
print("FINAL MODEL SAMPLE AND SPLIT")
print(f"Modeling observations: {len(model_data):,}")
print(f"Training set: {len(train_df):,} ({len(train_df) / len(model_data):.1%})")
print(f"Held-out test set: {len(test_df):,} ({len(test_df) / len(model_data):.1%})")
print("Split: 80% train / 20% test; random_state=42")

In [ ]:
# --------------------------------------------------------------------------
# 4.3 Fit the multiple linear regression
# --------------------------------------------------------------------------
# ols: ordinary least squares fit (used for R-squared and the diagnostic tests).
ols = sm.OLS(y_train, X_train).fit()

# robust_model: same coefficients, but HC3 robust standard errors, which stay
# valid when the residual variance is not constant. Used for all inference.
robust_model = sm.OLS(y_train, X_train).fit(cov_type="HC3")

# Predict polarization for the held-out reviews.
test_predictions = robust_model.predict(X_test)

# Baseline for comparison: always predict the training-set average polarization.
baseline_predictions = np.repeat(y_train.mean(), len(y_test))


def metric_row(actual, predicted):
    """MAE, RMSE and R-squared for one set of predictions."""
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R_squared": r2_score(actual, predicted),
    }


# Held-out performance of the model vs. the baseline (printed in the next cell).
performance_table = pd.DataFrame(
    {
        "Final regression": metric_row(y_test, test_predictions),
        "Training-mean baseline": metric_row(y_test, baseline_predictions),
    }
).T

In [ ]:
# --------------------------------------------------------------------------
# 4.4 Coefficient table and model performance
# --------------------------------------------------------------------------
# Standardized coefficients (all variables rescaled to mean 0, SD 1) let us
# compare the size of each predictor's effect on the same scale.
X_train_z = train_df[features].apply(lambda s: (s - s.mean()) / s.std(ddof=0))
y_train_z = (y_train - y_train.mean()) / y_train.std(ddof=0)
standardized_model = sm.OLS(
    y_train_z,
    sm.add_constant(X_train_z, has_constant="add"),
).fit(cov_type="HC3")

# One row per predictor: estimate, robust SE, t, p-value, 95% CI, standardized beta.
confidence_intervals = np.asarray(robust_model.conf_int())
coefficient_table = pd.DataFrame(
    {
        "coefficient": np.asarray(robust_model.params),
        "HC3_robust_SE": np.asarray(robust_model.bse),
        "t_value": np.asarray(robust_model.tvalues),
        "p_value": np.asarray(robust_model.pvalues),
        "CI_95_lower": confidence_intervals[:, 0],
        "CI_95_upper": confidence_intervals[:, 1],
        "standardized_beta": [np.nan, *np.asarray(standardized_model.params)[1:]],
    },
    index=X_train.columns,
)

# OUTPUT: coefficient table, held-out performance vs. baseline, and overall fit.
print("\nCOEFFICIENT TABLE — HC3 ROBUST STANDARD ERRORS")
display(coefficient_table.round(5))
print("\nHELD-OUT PERFORMANCE AND BASELINE COMPARISON")
display(performance_table.round(5))
print(f"Training R-squared: {ols.rsquared:.5f}")
print(f"Adjusted training R-squared: {ols.rsquared_adj:.5f}")
print(f"Overall F-test p-value: {ols.f_pvalue:.4g}")

In [ ]:
# --------------------------------------------------------------------------
# 4.5 Multicollinearity and model-assumption checks
# --------------------------------------------------------------------------
# Variance inflation factors: values above ~5 would mean the predictors overlap
# too much (e.g. word count and grade level measuring the same thing).
vif_design = sm.add_constant(train_df[features], has_constant="add")
vif_table = pd.DataFrame(
    {
        "predictor": features,
        "VIF": [
            variance_inflation_factor(vif_design.to_numpy(), i)
            for i in range(1, len(features) + 1)
        ],
    }
)

# Diagnostic tests on the training residuals:
#   Breusch-Pagan  -> is the residual variance constant? (p < 0.05 = no)
#   Jarque-Bera    -> are the residuals normally distributed? (p < 0.05 = no)
#   Ramsey RESET   -> is a straight-line relationship adequate? (p < 0.05 = no)
#   Durbin-Watson  -> are residuals independent? (close to 2 = yes)
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(ols.resid, X_train)
jb_stat, jb_p, residual_skew, residual_kurtosis = jarque_bera(ols.resid)
reset_test = linear_reset(ols, power=2, use_f=True)

diagnostic_table = pd.DataFrame(
    {
        "diagnostic": [
            "Breusch-Pagan LM",
            "Breusch-Pagan F",
            "Jarque-Bera normality",
            "Ramsey RESET linearity",
            "Durbin-Watson",
        ],
        "statistic": [
            bp_lm,
            bp_f,
            jb_stat,
            float(reset_test.fvalue),
            durbin_watson(ols.resid),
        ],
        "p_value": [bp_lm_p, bp_f_p, jb_p, float(reset_test.pvalue), np.nan],
    }
)

# OUTPUT: VIF table and diagnostics table.
print("\nVARIANCE INFLATION FACTORS")
display(vif_table.round(3))
print("\nREGRESSION DIAGNOSTICS")
display(diagnostic_table.round(5))
print(f"Residual skew: {residual_skew:.3f}; residual kurtosis: {residual_kurtosis:.3f}")

In [ ]:
# --------------------------------------------------------------------------
# 4.6 Model and diagnostic graphs
# --------------------------------------------------------------------------
# OUTPUT: 2x2 figure (saved as model_diagnostics.png):
#   top-left     residuals vs. fitted values (checks linearity / equal variance)
#   top-right    normal Q-Q plot of residuals (checks normality)
#   bottom-left  observed vs. predicted polarization, colored by critic status
#   bottom-right each coefficient with its 95% confidence interval
test_residuals = y_test - test_predictions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(x=test_predictions, y=test_residuals, alpha=0.15, s=12, ax=axes[0, 0])
axes[0, 0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0, 0].set(
    title="Residuals vs. Fitted Values (Held-out Test Set)",
    xlabel="Predicted polarization",
    ylabel="Residual",
)

# A random sample of 5,000 residuals keeps the Q-Q plot readable.
qq_residuals = test_residuals.sample(n=min(5_000, len(test_residuals)), random_state=42)
sm.qqplot(qq_residuals, line="45", ax=axes[0, 1])
axes[0, 1].set_title("Normal Q-Q Plot of Test Residuals")

plot_indices = test_df.sample(n=min(5_000, len(test_df)), random_state=42).index
sns.scatterplot(
    x=y_test.loc[plot_indices],
    y=test_predictions.loc[plot_indices],
    hue=test_df.loc[plot_indices, "is_top_critic"].map(
        {0: "Other critic", 1: "Top critic"}
    ),
    alpha=0.25,
    s=16,
    ax=axes[1, 0],
)
plot_min, plot_max = y_test.min(), y_test.max()
axes[1, 0].plot([plot_min, plot_max], [plot_min, plot_max], "k--", linewidth=1)
axes[1, 0].set(
    title="Observed vs. Predicted Polarization",
    xlabel="Observed polarization",
    ylabel="Predicted polarization",
)
axes[1, 0].legend(title="Critic status")

plot_coefficients = coefficient_table.drop(index="const").sort_values("coefficient")
axes[1, 1].errorbar(
    plot_coefficients["coefficient"],
    plot_coefficients.index,
    xerr=[
        plot_coefficients["coefficient"] - plot_coefficients["CI_95_lower"],
        plot_coefficients["CI_95_upper"] - plot_coefficients["coefficient"],
    ],
    fmt="o",
    color="#d65f8d",
    ecolor="#4e79a7",
    capsize=4,
)
axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1, 1].set(
    title="Regression Coefficients with 95% CIs",
    xlabel="Polarization units (0–0.5 scale)",
)

plt.tight_layout()
plt.savefig("model_diagnostics.png", dpi=150)
plt.show()

In [ ]:
# --------------------------------------------------------------------------
# 4.7 Compare the result with the team's R-squared goal
# --------------------------------------------------------------------------
final_r2 = performance_table.loc["Final regression", "R_squared"]
baseline_r2 = performance_table.loc["Training-mean baseline", "R_squared"]

# OUTPUT: held-out R-squared and whether the predetermined goal (R² >= 0.20) was met.
print("\n — MS3 MODEL CONCLUSION")
print(f"Held-out R-squared: {final_r2:.5f}")
print(f"Baseline held-out R-squared: {baseline_r2:.5f}")
print(f"Predetermined R-squared >= 0.20 goal met: {final_r2 >= 0.20}")
print("Statistical significance must be interpreted separately from predictive usefulness.")
print("These findings are associative and do not support a causal conclusion.")

---
## Section 5: Model testing

Apply the finalized regression equation to the held-out 20% of reviews and check it against the plan's evaluation criteria: R² above 0, MAE and RMSE below the training-mean baseline, and the R² > 0.20 and ±0.5 accuracy goals.

**Outputs:** the regression equation; performance, generalization, criteria and margin-of-error tables; a conclusion; and `model_testing.png`.

In [ ]:
# Reload the feature dataset saved in Section 3 and confirm the required
# columns are present. (df already holds the same data if you ran every cell
# in order. This check makes sure the file Section 3 saved is complete.)
import os
import pandas as pd

FEATURE_DATA_PATH = "/content/rotten_tomatoes_data_with_vars.csv"
assert os.path.exists(FEATURE_DATA_PATH), "Feature CSV is missing — run Section 3 first."
df = pd.read_csv(FEATURE_DATA_PATH)

# OUTPUT: size of the reloaded data and a True/False check for each required column.
print(f"Loaded feature dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Required columns present:", {
    c: c in df.columns
    for c in ["reviewId", "isTopCritic", "polarization", "word_count", "grade_level"]
})

In [ ]:
# Camila

# --------------------------------------------------------------------------
# 5.0 Packages and the plan's success thresholds
# --------------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

R2_GOAL = 0.20          # plan: R² > 20%
TOLERANCE = 0.5         # plan: "margin of error of ± 0.5" (see warning in 5.3)
ACCURACY_GOAL = 0.70    # plan: 70% accuracy

# Make sure Section 4 was run, since this section reuses its model and split.
for name in ["robust_model", "X_train", "X_test", "y_train", "y_test"]:
    if name not in globals():
        raise NameError(f"'{name}' not found — run Zichen's model cell first.")

final_model = robust_model
print("CAMILA — MODEL TESTING")
print(f"Training set: {len(y_train):,} | Held-out set (not used in fitting): {len(y_test):,}")

# OUTPUT: the finalized regression equation with its fitted coefficients.
b = final_model.params
print("\nFINALIZED REGRESSION EQUATION (fit on training data):")
print(f"polarization = {b['const']:.4f} "
      f"+ {b['log_word_count']:.4f}*log(1 + word_count) "
      f"+ {b['grade_level_model']:.4f}*grade_level "
      f"+ {b['is_top_critic']:.4f}*top_critic")

# --------------------------------------------------------------------------
# 5.1 Performance metrics: model vs. baseline, in-sample and held-out
# --------------------------------------------------------------------------
train_mean = y_train.mean()   # baseline uses ONLY the training mean

pred_train = final_model.predict(X_train)
pred_test = final_model.predict(X_test)      # apply equation to new reviews
base_train = np.full(len(y_train), train_mean)
base_test = np.full(len(y_test), train_mean)


def metrics(actual, predicted):
    """R², MAE, RMSE and share of predictions within ±TOLERANCE."""
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    return {
        "R_squared": r2_score(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        f"Accuracy_±{TOLERANCE}": np.mean(np.abs(actual - predicted) <= TOLERANCE),
    }


performance = pd.DataFrame(
    {
        ("In-sample (training)", "Final regression"): metrics(y_train, pred_train),
        ("In-sample (training)", "Intercept-only baseline"): metrics(y_train, base_train),
        ("Held-out (new data)", "Final regression"): metrics(y_test, pred_test),
        ("Held-out (new data)", "Training-mean baseline"): metrics(y_test, base_test),
    }
).T

# OUTPUT: performance table (model and baseline, training and held-out).
print("\nPERFORMANCE METRICS: MODEL vs. BASELINE")
display(performance.round(5))

train_row = performance.loc[("In-sample (training)", "Final regression")]
test_row = performance.loc[("Held-out (new data)", "Final regression")]
train_base = performance.loc[("In-sample (training)", "Intercept-only baseline")]
test_base = performance.loc[("Held-out (new data)", "Training-mean baseline")]

# Generalization check: a held-out result close to the training result means
# the model is not overfitting the training reviews.
generalization = pd.DataFrame(
    {"Training": train_row[["R_squared", "MAE", "RMSE"]],
     "Held-out": test_row[["R_squared", "MAE", "RMSE"]]}
)
generalization["Difference (held-out − training)"] = (
    generalization["Held-out"] - generalization["Training"]
)

# OUTPUT: generalization table.
print("\nTRAINING vs. HELD-OUT RESULTS (generalization check)")
display(generalization.round(5))

# --------------------------------------------------------------------------
# 5.2 Check each criterion from the analysis plan
# --------------------------------------------------------------------------
def yes_no(condition):
    return "YES" if condition else "NO"


criteria = pd.DataFrame(
    [
        ("In-sample", "R² > 0", yes_no(train_row["R_squared"] > 0)),
        ("In-sample", "MAE below intercept-only baseline", yes_no(train_row["MAE"] < train_base["MAE"])),
        ("In-sample", "RMSE below intercept-only baseline", yes_no(train_row["RMSE"] < train_base["RMSE"])),
        ("Held-out", "R² > 0", yes_no(test_row["R_squared"] > 0)),
        ("Held-out", "MAE below training-mean baseline", yes_no(test_row["MAE"] < test_base["MAE"])),
        ("Held-out", "RMSE below training-mean baseline", yes_no(test_row["RMSE"] < test_base["RMSE"])),
        ("Goal", f"Held-out R² > {R2_GOAL}", yes_no(test_row["R_squared"] > R2_GOAL)),
        ("Goal", f"≥{ACCURACY_GOAL:.0%} of predictions within ±{TOLERANCE}",
         yes_no(test_row[f"Accuracy_±{TOLERANCE}"] >= ACCURACY_GOAL)),
    ],
    columns=["stage", "criterion", "met"],
)

# OUTPUT: YES/NO for each plan criterion.
print("\nPLAN CRITERIA CHECK")
display(criteria)
print(f"R² gap (training − held-out): {train_row['R_squared'] - test_row['R_squared']:.5f} "
      "(a small gap means the model generalizes beyond the training sample)")

# --------------------------------------------------------------------------
# 5.3 Margin-of-error accuracy (context for the ±0.5 goal)
# --------------------------------------------------------------------------
if TOLERANCE >= 0.5:
    print(f"\nWARNING: polarization only ranges from 0 to 0.5, so a margin of ±{TOLERANCE} "
          "is met by almost any prediction. Accuracy at tighter margins is shown below.")

margins = [0.05, 0.10, 0.25, TOLERANCE]
margin_table = pd.DataFrame(
    {
        "Final regression": [np.mean(np.abs(y_test - pred_test) <= m) for m in margins],
        "Training-mean baseline": [np.mean(np.abs(y_test - base_test) <= m) for m in margins],
    },
    index=[f"within ±{m}" for m in margins],
)

# OUTPUT: share of held-out predictions within each margin of error.
print("\nHELD-OUT ACCURACY AT DIFFERENT MARGINS OF ERROR")
display(margin_table.style.format("{:.1%}"))

# --------------------------------------------------------------------------
# 5.4 Graphs for the report
# --------------------------------------------------------------------------
# OUTPUT: 2x2 figure (saved as model_testing.png):
#   (a) residuals vs. fitted on held-out data
#   (b) observed vs. predicted on held-out data, with the baseline line
#   (c) MAE and RMSE for model vs. baseline, training vs. held-out
#   (d) training and held-out R² compared with the 0.20 goal
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
test_resid = y_test - pred_test

# (a) Residuals vs. fitted on the new data
sns.scatterplot(x=pred_test, y=test_resid, alpha=0.15, s=10, ax=axes[0, 0])
axes[0, 0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0, 0].set(title="Residuals vs. Fitted (Held-out Data)",
               xlabel="Predicted polarization", ylabel="Residual")

# (b) Observed vs. predicted on the new data
sample_idx = y_test.sample(n=min(5_000, len(y_test)), random_state=42).index
sns.scatterplot(x=y_test.loc[sample_idx], y=pred_test.loc[sample_idx],
                alpha=0.2, s=10, ax=axes[0, 1])
axes[0, 1].plot([0, 0.5], [0, 0.5], "k--", linewidth=1, label="Perfect prediction")
axes[0, 1].axhline(train_mean, color="#d65f8d", linestyle=":", label="Baseline (training mean)")
axes[0, 1].set(title="Observed vs. Predicted Polarization (Held-out Data)",
               xlabel="Observed polarization", ylabel="Predicted polarization")
axes[0, 1].legend()

# (c) MAE and RMSE: model vs. baseline, training vs. held-out
err_plot = performance[["MAE", "RMSE"]].reset_index()
err_plot.columns = ["data", "model", "MAE", "RMSE"]
err_plot = err_plot.melt(id_vars=["data", "model"], var_name="metric", value_name="value")
err_plot["model"] = err_plot["model"].replace(
    {"Intercept-only baseline": "Baseline", "Training-mean baseline": "Baseline"}
)
err_plot["group"] = err_plot["metric"] + "\n" + err_plot["data"]
sns.barplot(data=err_plot, x="group", y="value", hue="model", ax=axes[1, 0])
axes[1, 0].set(title="Prediction Error: Model vs. Baseline", xlabel="", ylabel="Error (polarization units)")
axes[1, 0].legend(title="")

# (d) R²: training vs. held-out vs. the goal
axes[1, 1].bar(["Training", "Held-out"],
               [train_row["R_squared"], test_row["R_squared"]], color="#4e79a7")
axes[1, 1].axhline(R2_GOAL, color="#d65f8d", linestyle="--", label=f"Goal R² > {R2_GOAL}")
axes[1, 1].axhline(0, color="black", linewidth=1)
axes[1, 1].set(title="R²: Training vs. Held-out", ylabel="R²")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("model_testing.png", dpi=150)
plt.show()

# --------------------------------------------------------------------------
# 5.5 Conclusion
# --------------------------------------------------------------------------
# OUTPUT: one-line summary of each key result.
print("\nCAMILA — MODEL TESTING CONCLUSION")
print(f"Training R²: {train_row['R_squared']:.5f} | Held-out R²: {test_row['R_squared']:.5f}")
print(f"Held-out MAE: model {test_row['MAE']:.5f} vs. baseline {test_base['MAE']:.5f}")
print(f"Held-out RMSE: model {test_row['RMSE']:.5f} vs. baseline {test_base['RMSE']:.5f}")
beats_baseline = (test_row["R_squared"] > 0 and test_row["MAE"] < test_base["MAE"]
                  and test_row["RMSE"] < test_base["RMSE"])
print(f"Model beats the training-mean baseline on new data: {yes_no(beats_baseline)}")
print(f"R² goal (> {R2_GOAL}) met: {yes_no(test_row['R_squared'] > R2_GOAL)}")

---
## Export all outputs

Collect every table (as CSV) and graph (as PNG) from Sections 3-5 into an `OUTPUT` folder, then download it as one zip file.

**Output:** `OUTPUT.zip`, downloaded to your computer.

In [ ]:
import glob
import os
import shutil

import pandas as pd
from google.colab import files

os.makedirs("OUTPUT", exist_ok=True)
saved = []

# Tables from Sections 4 and 5 -> one CSV each.
table_names = [
    "coefficient_table",   # Zichen: coefficients, robust SEs, p-values, CIs
    "performance_table",   # Zichen: held-out performance vs. baseline
    "vif_table",           # Zichen: variance inflation factors
    "diagnostic_table",    # Zichen: regression diagnostics
    "performance",         # Camila: in-sample and held-out metrics vs. baseline
    "generalization",      # Camila: training vs. held-out comparison
    "criteria",            # Camila: plan criteria check
    "margin_table",        # Camila: accuracy at different margins of error
]
for name in table_names:
    obj = globals().get(name)
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(f"OUTPUT/{name}.csv")
        saved.append(f"{name}.csv")

# Summary statistics from Section 3 (Arya's EDA).
eda_cols = ["normalized_score", "polarization", "word_count", "grade_level"]
if "df" in globals() and all(c in df.columns for c in eda_cols):
    df[eda_cols].describe().to_csv("OUTPUT/summary_statistics.csv")
    saved.append("summary_statistics.csv")

# Graphs saved with plt.savefig in Sections 3-5.
for image in glob.glob("*.png"):
    shutil.copy(image, "OUTPUT/")
    saved.append(image)

# OUTPUT: list of saved files, then OUTPUT.zip is downloaded.
print("Saved to OUTPUT/:")
for f in sorted(saved):
    print("  -", f)

shutil.make_archive("OUTPUT", "zip", "OUTPUT")
files.download("OUTPUT.zip")